# Phase 2: Energy-Based Frequency Masking Visualization

This notebook visualizes Phase 2 Energy-Based Frequency Masking, which extends random masking by preferentially targeting low-energy frequency bands.

## Key Features
- Preserves high-energy regions (likely containing calls)
- Masks low-energy regions (likely noise)
- Uses EMA for stable energy tracking
- Supports hybrid mode (energy + random weighting)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

print(f"Project root: {project_root}")
print(f"Source path: {src_path}")
print("Setup complete!")

## 2. Configuration

In [ ]:
from datamodules.components.frequency_masker import EnergyBasedFrequencyMasker

masker = EnergyBasedFrequencyMasker(
    n_bands=2,
    mask_prob=1.0,
    max_band_width=16,
    n_mels=128,
    energy_weight=0.8,
    ema_alpha=0.1
)

print("EnergyBasedFrequencyMasker initialized!")
print(f"Config: {masker}")

## 3. Test Data Generation

In [ ]:
def create_energy_pattern_spectrogram(pattern='low_top_high_bottom', shape=(100, 128)):
    """Create synthetic spectrogram with specific energy patterns."""
    time_steps, n_freq = shape
    
    if pattern == 'low_top_high_bottom':
        spec = np.zeros(shape)
        spec[:, :n_freq//3] = np.random.randn(time_steps, n_freq//3) * 0.2 + 1.5
        spec[:, n_freq//3:2*n_freq//3] = np.random.randn(time_steps, n_freq//3) * 0.2 + 0.8
        spec[:, 2*n_freq//3:] = np.random.randn(time_steps, n_freq//3) * 0.1 + 0.3
    elif pattern == 'high_top_low_bottom':
        spec = np.zeros(shape)
        spec[:, :n_freq//3] = np.random.randn(time_steps, n_freq//3) * 0.1 + 0.3
        spec[:, n_freq//3:2*n_freq//3] = np.random.randn(time_steps, n_freq//3) * 0.2 + 0.8
        spec[:, 2*n_freq//3:] = np.random.randn(time_steps, n_freq//3) * 0.2 + 1.5
    elif pattern == 'mixed_energy':
        spec = np.zeros(shape)
        band_width = n_freq // 8
        for i in range(8):
            start = i * band_width
            end = min((i + 1) * band_width, n_freq)
            if i % 2 == 0:
                spec[:, start:end] = np.random.randn(time_steps, end-start) * 0.2 + 1.2
            else:
                spec[:, start:end] = np.random.randn(time_steps, end-start) * 0.1 + 0.4
    else:  # uniform
        spec = np.random.randn(*shape) * 0.3 + 0.8
    
    return np.clip(spec, 0, 2)

test_patterns = ['low_top_high_bottom', 'high_top_low_bottom', 'mixed_energy', 'uniform']
test_specs = {p: create_energy_pattern_spectrogram(p) for p in test_patterns}
print(f"Generated {len(test_specs)} test spectrograms")

## 4. Visualization Functions

### Function 1: Energy Profile

In [ ]:
def visualize_energy_profile(spec, masker, title="Energy Profile"):
    """Visualize band energies with masking weights."""
    band_energies = masker._compute_band_energies(spec)
    masker._update_energy_ema(band_energies)
    masking_weights = masker._compute_masking_weights(masker.band_energy_ema)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    n_bands = len(band_energies)
    band_indices = np.arange(n_bands)
    
    ax1.bar(band_indices, band_energies, color='blue', alpha=0.7, edgecolor='black')
    ax1.set_xlabel('Band Index')
    ax1.set_ylabel('Energy')
    ax1.set_title('Band Energies', fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    ax2.bar(band_indices, masking_weights, color='red', alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Band Index')
    ax2.set_ylabel('Masking Weight')
    ax2.set_title('Masking Weights (Low Energy = High Weight)', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Energy - Mean: {band_energies.mean():.3f}, Std: {band_energies.std():.3f}")
    print(f"Highest weight band: {np.argmax(masking_weights)} (weight={masking_weights.max():.3f})")
    stats = masker.get_energy_stats()
    print(f"Samples seen: {stats['n_samples_seen']}, EMA alpha: {stats['ema_alpha']}")

print("Function 1 defined!")

### Function 2: Before/After with Energy

In [ ]:
def visualize_before_after_with_energy(original, masked, band_energies, title="Energy-Based Masking"):
    """3-panel view: original, masked, energy profile."""
    difference = np.abs(original - masked)
    mask_binary = (difference > 0.01).astype(float)
    
    fig = plt.figure(figsize=(16, 4))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 0.6])
    
    ax1 = fig.add_subplot(gs[0, 0])
    im1 = ax1.imshow(original.T, aspect='auto', origin='lower', cmap='viridis')
    ax1.set_title('Original', fontweight='bold')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Frequency')
    plt.colorbar(im1, ax=ax1)
    
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(masked.T, aspect='auto', origin='lower', cmap='viridis')
    ax2.imshow(mask_binary.T, aspect='auto', origin='lower', cmap='Reds', alpha=0.3)
    ax2.set_title('Masked (Red = Masked)', fontweight='bold')
    ax2.set_xlabel('Time')
    ax2.set_ylabel('Frequency')
    
    ax3 = fig.add_subplot(gs[0, 2])
    n_bands = len(band_energies)
    ax3.barh(np.arange(n_bands), band_energies, color='blue', alpha=0.7)
    ax3.set_ylabel('Band Index')
    ax3.set_xlabel('Energy')
    ax3.set_title('Band Energies', fontweight='bold')
    ax3.invert_yaxis()
    ax3.grid(True, alpha=0.3, axis='x')
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Masked: {np.sum(mask_binary)/mask_binary.size:.2%}")

print("Function 2 defined!")

### Function 3: EMA Evolution

In [ ]:
def visualize_ema_evolution(masker, spec_sequence, title="EMA Evolution"):
    """Track EMA over multiple batches."""
    masker.band_energy_ema = None
    masker.n_samples_seen = 0
    
    ema_history = []
    energy_history = []
    
    print(f"Processing {len(spec_sequence)} batches...")
    for i, spec in enumerate(spec_sequence):
        band_energies = masker._compute_band_energies(spec)
        energy_history.append(band_energies.copy())
        masker._update_energy_ema(band_energies)
        ema_history.append(masker.band_energy_ema.copy())
    
    ema_history = np.array(ema_history)
    energy_history = np.array(energy_history)
    
    n_bands = ema_history.shape[1]
    fig, axes = plt.subplots(n_bands, 1, figsize=(12, 2.5*n_bands))
    if n_bands == 1:
        axes = [axes]
    
    batch_indices = np.arange(len(spec_sequence))
    
    for band_idx in range(n_bands):
        ax = axes[band_idx]
        ax.plot(batch_indices, energy_history[:, band_idx], 
               color='lightblue', linewidth=1, alpha=0.5, label='Raw Energy')
        ax.plot(batch_indices, ema_history[:, band_idx],
               color='blue', linewidth=2, label='EMA')
        ax.set_xlabel('Batch')
        ax.set_ylabel('Energy')
        ax.set_title(f'Band {band_idx} (alpha={masker.ema_alpha})', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Processed {len(spec_sequence)} batches")

print("Function 3 defined!")

### Function 4: Hybrid Mode Comparison

In [ ]:
def visualize_hybrid_mode_comparison(spec, energy_weights=[0.0, 0.25, 0.5, 0.75, 1.0]):
    """Compare different energy_weight values."""
    n_weights = len(energy_weights)
    fig, axes = plt.subplots(1, n_weights, figsize=(4 * n_weights, 4))
    if n_weights == 1:
        axes = [axes]
    
    for i, energy_weight in enumerate(energy_weights):
        masker = EnergyBasedFrequencyMasker(
            n_bands=2, mask_prob=1.0, max_band_width=16,
            n_mels=128, energy_weight=energy_weight, ema_alpha=0.1
        )
        masked = masker(spec)
        difference = np.abs(spec - masked)
        mask_binary = (difference > 0.01).astype(float)
        
        ax = axes[i]
        ax.imshow(masked.T, aspect='auto', origin='lower', cmap='viridis')
        ax.imshow(mask_binary.T, aspect='auto', origin='lower', cmap='Reds', alpha=0.3)
        
        masked_frac = np.sum(mask_binary) / mask_binary.size
        mode = "Random" if energy_weight == 0.0 else "Energy" if energy_weight == 1.0 else "Hybrid"
        ax.set_title(f'weight={energy_weight}\n{mode}\n{masked_frac:.1%} masked', fontweight='bold')
        ax.set_xlabel('Time')
        ax.set_ylabel('Frequency')
    
    plt.suptitle('Hybrid Mode Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Function 4 defined!")

## 5. Example Usage

### 5.1 Energy Profiles

In [ ]:
for pattern_name, spec in test_specs.items():
    print(f"\n{'='*50}\nPattern: {pattern_name}\n{'='*50}")
    masker = EnergyBasedFrequencyMasker(
        n_bands=2, mask_prob=1.0, max_band_width=16,
        energy_weight=0.8, ema_alpha=0.1
    )
    visualize_energy_profile(spec, masker, f"Energy Profile: {pattern_name}")

### 5.2 Before/After Comparison

In [ ]:
for pattern_name, spec in test_specs.items():
    print(f"\nPattern: {pattern_name}")
    masker = EnergyBasedFrequencyMasker(
        n_bands=2, mask_prob=1.0, max_band_width=16,
        energy_weight=0.8, ema_alpha=0.1
    )
    band_energies = masker._compute_band_energies(spec)
    masked = masker(spec)
    visualize_before_after_with_energy(spec, masked, band_energies, f"Masking: {pattern_name}")

### 5.3 EMA Evolution

In [ ]:
spec_sequence = []
for _ in range(10):
    spec_sequence.append(create_energy_pattern_spectrogram('low_top_high_bottom'))
for _ in range(10):
    spec_sequence.append(create_energy_pattern_spectrogram('high_top_low_bottom'))
for _ in range(10):
    spec_sequence.append(create_energy_pattern_spectrogram('mixed_energy'))

masker = EnergyBasedFrequencyMasker(
    n_bands=2, mask_prob=1.0, max_band_width=16,
    energy_weight=0.8, ema_alpha=0.1
)
visualize_ema_evolution(masker, spec_sequence, "EMA Evolution: Changing Patterns")

### 5.4 Hybrid Mode

In [ ]:
test_spec = test_specs['low_top_high_bottom']
visualize_hybrid_mode_comparison(test_spec)

## 6. Analysis and Insights

### Key Observations

1. **Energy-Aware Selection**: Phase 2 preferentially targets low-energy bands (high masking weights for low energy)
2. **Adaptive Behavior**: Automatically adjusts to each spectrogram's energy distribution
3. **EMA Smoothing**: Provides stable estimates, preventing erratic masking
4. **Hybrid Flexibility**: energy_weight controls random vs energy-based trade-off

### Comparison with Phase 1

| Aspect | Phase 1 (Random) | Phase 2 (Energy) |
|--------|------------------|------------------|
| Selection | Uniform | Weighted by energy |
| Adaptation | None | Per-spectrogram |
| High-energy | May mask | Preserved |
| Low-energy | Equal prob | Preferentially masked |

### Advantages
- Intelligent: targets less informative regions
- Preserves important features (high-energy calls)
- Adaptive to different species/conditions
- Stable via EMA

### Limitations
- Energy ≠ importance (quiet calls exist)
- No model feedback
- Computational overhead

These limitations motivate Phase 3 (Activation-Guided) masking.

## 7. Custom Analysis

In [ ]:
# Custom analysis cell 1
# Example: Test different ema_alpha values
# for alpha in [0.05, 0.1, 0.2]:
#     masker = EnergyBasedFrequencyMasker(n_bands=2, mask_prob=1.0, 
#                                         max_band_width=16, energy_weight=0.8, ema_alpha=alpha)
#     visualize_ema_evolution(masker, spec_sequence[:15], f"EMA (alpha={alpha})")

In [ ]:
# Custom analysis cell 2
# Add your own experiments here